# 优化模型参数

In [6]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="../data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="../data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

### 超参数

超参数是可调节的参数，用于控制模型优化过程。不同的超参数值会影响模型的训练和收敛速度

- 迭代次数（Number of Epochs） ——遍历数据集的次数。
- 批大小——在参数更新之前，通过网络传播的数据样本数量。
- 学习率——指每个批次/周期中模型参数的更新量。较小的学习率会导致学习速度慢，而较大的学习率则可能导致训练过程中出现不可预测的行为。


In [7]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

### 优化循环

一旦我们设定好超参数，就可以使用优化循环来训练和优化模型。优化循环的每次迭代称为一个周期（epoch）。

每个纪元由两个主要部分组成：
- 训练循环- 遍历训练数据集，并尝试收敛到最优参数。
- 验证/测试循环- 遍历测试数据集，检查模型性能是否有所提高

### 损失函数

当输入一些训练数据时，未经训练的网络很可能无法给出正确答案。损失函数衡量的是所得结果与目标值的差异程度，而这正是我们在训练过程中需要最小化的目标。为了计算损失，我们使用给定数据样本的输入进行预测，并将其与真实数据标签值进行比较。

常用的损失函数包括用于回归任务的 `nn.MSELoss`（均方误差）和 用于分类任务的 `nn.NLLLoss`（负对数似然）。`nn.CrossEntropyLoss` 结合了`nn.LogSoftmax`和`nn.NLLLoss`


In [8]:
# 初始化损失函数
loss_fn = nn.CrossEntropyLoss()

### 优化器

优化是指在每个训练步骤中调整模型参数以减少模型误差的过程

In [9]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

在训练循环中，优化过程分为三个步骤：
- 调用此函数optimizer.zero_grad()重置模型参数的梯度。默认情况下，梯度会累加；为了防止重复计数，我们在每次迭代中显式地将其归零。
- 使用调用反向传播预测损失函数loss.backward()。PyTorch 会将损失函数相对于每个参数的梯度存储起来。
- 一旦我们得到梯度，我们就调用函数，optimizer.step()根据反向传播中收集到的梯度来调整参数。

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # 计算预测值
        pred = model(X)
        loss = loss_fn(pred, y)

        # 反向传播
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [11]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.300914  [   64/60000]
loss: 2.293150  [ 6464/60000]
loss: 2.268248  [12864/60000]
loss: 2.260205  [19264/60000]
loss: 2.253774  [25664/60000]
loss: 2.203906  [32064/60000]
loss: 2.219587  [38464/60000]
loss: 2.182977  [44864/60000]
loss: 2.183802  [51264/60000]
loss: 2.135195  [57664/60000]
Test Error: 
 Accuracy: 44.8%, Avg loss: 2.138286 

Epoch 2
-------------------------------
loss: 2.157682  [   64/60000]
loss: 2.148163  [ 6464/60000]
loss: 2.080240  [12864/60000]
loss: 2.092175  [19264/60000]
loss: 2.043347  [25664/60000]
loss: 1.968403  [32064/60000]
loss: 2.001096  [38464/60000]
loss: 1.920337  [44864/60000]
loss: 1.935265  [51264/60000]
loss: 1.840611  [57664/60000]
Test Error: 
 Accuracy: 57.9%, Avg loss: 1.842011 

Epoch 3
-------------------------------
loss: 1.894320  [   64/60000]
loss: 1.854745  [ 6464/60000]
loss: 1.728511  [12864/60000]
loss: 1.763423  [19264/60000]
loss: 1.652223  [25664/60000]
loss: 1.608064  [32064/600